# Can Image Diffusion Policy (Light)

This notebook keeps the official Diffusion Policy image Colab flow, but the large code cells are split into small Python files.

- `can_policy.py`: ResNet image encoder + ConditionalUnet1D
- `can_data.py`: Can HDF5 dataset + dataset preparation
- `can_teleop.py`: SpaceMouse teleop and demo collection
- `train_can_image.py`: script version of the training loop
- `eval_can_image.py`: robosuite PickPlaceCan rollout evaluation


## 1. Imports

In [ ]:
from pathlib import Path
import torch

from can_data import CanImageDataset, CAMERA_KEYS
from can_policy import make_policy

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())

## 2. Dataset

The light version uses our Can HDF5 dataset directly. No robomimic replay buffer, no zarr cache, no Hydra.

In [ ]:
dataset_path = 'data/robomimic/datasets/can/custom/image.hdf5'
pred_horizon = 16
obs_horizon = 2
action_horizon = 8

dataset = CanImageDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon,
    camera_keys=CAMERA_KEYS,
)

sample = dataset[0]
print('dataset length:', len(dataset))
for key, value in sample.items():
    print(key, tuple(value.shape), value.dtype)

## 3. Network Smoke Test

This is the official Colab architecture adapted to two Can cameras and low-dimensional proprioception.

In [ ]:
action_dim = sample['action'].shape[-1]
lowdim_dim = sample['lowdim'].shape[-1]
nets = make_policy(obs_horizon=obs_horizon, action_dim=action_dim, lowdim_dim=lowdim_dim)

batch = {
    'lowdim': sample['lowdim'].unsqueeze(0),
}
for key in CAMERA_KEYS:
    batch[key] = sample[key].unsqueeze(0)

with torch.no_grad():
    cond = nets['obs_encoder'](batch).flatten(start_dim=1)
    noisy_action = torch.zeros(1, pred_horizon, action_dim)
    noise = nets['noise_pred_net'](noisy_action, torch.zeros(1, dtype=torch.long), cond)

print('global condition:', tuple(cond.shape))
print('noise prediction:', tuple(noise.shape))

## 4. Training

Use the script for maintainability. Start with a one-step smoke test, then run full training.

In [ ]:
# Smoke test
# !python train_can_image.py \
#   --dataset data/robomimic/datasets/can/custom/image.hdf5 \
#   --output data/outputs/can_image_light_debug \
#   --num-epochs 1 \
#   --max-train-steps 1 \
#   --batch-size 2 \
#   --num-workers 0 \
#   --device cuda:0

In [ ]:
# Full training
# !python train_can_image.py \
#   --dataset data/robomimic/datasets/can/custom/image.hdf5 \
#   --output data/outputs/can_image_light \
#   --device cuda:0

## 5. Evaluation

In [ ]:
# !python eval_can_image.py \
#   --checkpoint data/outputs/can_image_light/latest.pt \
#   --output data/eval_can_image_light \
#   --device cuda:0 \
#   --num-episodes 4

## 6. Teleoperation / Demo Collection

EEF downward orientation lock and Linux SpaceMouse handling are in `can_teleop.py`.

In [ ]:
# Manual teleop
# !python can_teleop.py --mode teleop

# Collect demos
# !python can_teleop.py --mode collect